In [1]:
# Install and import all required Python libraries
!pip install openpyxl scikit-learn -q

import os
import pandas as pd
import numpy as np
import smtplib
from datetime import date
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries ready!")

✅ All libraries ready!


In [2]:
#UPLOAD DATASET
from google.colab import files

# Upload diabetic_data.csv
print("Upload diabetic_data.csv:")
files.upload()
print("✅ Dataset uploaded!")

Upload diabetic_data.csv:


Saving diabetic_data.csv to diabetic_data (2).csv
✅ Dataset uploaded!


# Load the CSV into a pandas DataFrame and inspect its structure before any cleaning or processing.

LOAD & PREVIEW DATA

PURPOSE: Load the CSV into a pandas DataFrame and inspect its structure before any cleaning or processing.

WHAT WE CHECK:
Total records : How many patient encounters exist
Total columns : How many features are available
Readmission breakdown : Distribution of our target variable

KEY FINDINGS FROM OUTPUT:
101,766 total patient records across 50 columns
Target column 'readmitted' has 3 values:
NO   : 54,864 patients → not readmitted
>30  : 35,545 patients → readmitted after 30 days
<30  : 11,357 patients → readmitted within 30 days

WHY THIS MATTERS:
Only 11.1% of patients are readmitted within 30 days.
This severe class imbalance (91% vs 9%) is the core
challenge we must solve before training any ML model.

OUTPUT:
Total records  : 101,766
Total columns  : 50
Readmission Breakdown: NO=54864, >30=35545, <30=11357





In [3]:


df = pd.read_csv("diabetic_data.csv")

print(f"📋 Total records  : {len(df):,}")
print(f"📊 Total columns  : {len(df.columns)}")
print(f"\n--- Readmission Breakdown ---")
print(df['readmitted'].value_counts())

📋 Total records  : 101,766
📊 Total columns  : 50

--- Readmission Breakdown ---
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


In [4]:
# DATA CLEANING:
# PURPOSE:
# Clean the raw dataset to remove noise, handle missing values,
# fix data types, and create the binary target variable.
#
# CLEANING STEPS PERFORMED:
#
# 1. Replace '?' with NaN
#    The dataset uses '?' as a placeholder for missing values.
#    We convert these to NaN so pandas handles them correctly.
#
# 2. Drop 3 columns
#    - weight          : 97% missing — unusable
#    - payer_code      : Insurance code, not clinically relevant
#    - medical_specialty: 49% missing — too sparse
#
# 3. Remove duplicate patients
#    Same patient can have multiple hospital encounters.
#    We keep only the FIRST encounter to avoid data leakage
#    where later visits influence earlier predictions.
#
# 4. Drop missing race rows
#    Race has a small number of NaN values.
#    Dropped to ensure clean categorical encoding later.
#
# 5. Remove invalid gender entries
#    3 records have gender = 'Unknown/Invalid' — removed.
#
# 6. Convert age brackets to numeric midpoints
#    Original: '[60-70)' → Converted to: 65
#    This allows the model to treat age as a continuous
#    numeric feature rather than a string category.
#
# 7. Create binary target variable
#    readmitted_binary = 1 if readmitted < 30 days (HIGH RISK)
#    readmitted_binary = 0 if NO or >30 days  (LOW RISK)
#    Why? 30-day readmission is the Medicare/CMS benchmark
#    that hospitals are penalized for.
#
# OUTPUT:
# ✅ Unique Cleaned records : 69,569
# High Risk (<30 days) : 6,152 (8.8%)
# Low Risk             : 63,417 (91.2%)

data = df.copy()

# Replace '?' with NaN
data.replace('?', np.nan, inplace=True)

# Drop columns — weight 97% missing, payer/specialty not useful
data.drop(columns=['weight', 'payer_code', 'medical_specialty'], inplace=True)

# Keep first encounter per patient only
data.drop_duplicates(subset='patient_nbr', keep='first', inplace=True)

# Drop missing race rows
data.dropna(subset=['race'], inplace=True)

# Remove invalid gender
data = data[data['gender'] != 'Unknown/Invalid']

# Convert age brackets to numeric midpoints
age_map = {
    '[0-10)'  : 5,  '[10-20)' : 15, '[20-30)' : 25,
    '[30-40)' : 35, '[40-50)' : 45, '[50-60)' : 55,
    '[60-70)' : 65, '[70-80)' : 75, '[80-90)' : 85,
    '[90-100)': 95
}
data['age'] = data['age'].map(age_map)

# Target variable — 1 = readmitted within 30 days (HIGH RISK)
data['readmitted_binary'] = (data['readmitted'] == '<30').astype(int)

print(f"✅ Unique Cleaned records : {len(data):,}")
print(f"\nHigh Risk (<30 days) : {data['readmitted_binary'].sum():,} ({data['readmitted_binary'].mean()*100:.1f}%)")
print(f"Low Risk             : {(data['readmitted_binary']==0).sum():,} ({(1-data['readmitted_binary'].mean())*100:.1f}%)")

✅ Unique Cleaned records : 69,569

High Risk (<30 days) : 6,152 (8.8%)
Low Risk             : 63,417 (91.2%)


In [5]:
# Numerical features

# FEATURE ENGINEERING & ENCODING

# PURPOSE:
# Select the 20 most clinically meaningful features and convert
# all categorical text values into numbers the model can use.
#
# FEATURES SELECTED:
#
# Numerical (12) — used directly as numbers:
# - age                    : Patient age (converted to midpoint)
# - time_in_hospital       : Length of stay in days
# - num_lab_procedures     : Number of lab tests performed
# - num_procedures         : Number of medical procedures
# - num_medications        : Number of medications prescribed
# - number_outpatient      : Prior outpatient visits
# - number_emergency       : Prior emergency visits
# - number_inpatient       : Prior inpatient admissions
# - number_diagnoses       : Number of diagnoses recorded
# - admission_type_id      : Type of admission (emergency, elective)
# - discharge_disposition_id: Where patient went after discharge
# - admission_source_id    : How patient was admitted
#
# Categorical (8) — encoded with LabelEncoder:
# - race, gender, insulin, diabetesMed, change,
#   metformin, glipizide, glyburide
#
# ENCODING METHOD — LabelEncoder:
# Converts text categories to integers:
# Example: 'No' → 0, 'Steady' → 1, 'Up' → 2, 'Down' → 3
# Chosen because Random Forest does not assume numeric ordering
# unlike linear models — it splits on exact values.
#
# OUTPUT:
# ✅ Features ready  : 20
# 📊 Training size   : 69,569 records


num_features = [
    'age',
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

# Categorical features to encode
cat_features = [
    'race',
    'gender',
    'insulin',
    'diabetesMed',
    'change',
    'metformin',
    'glipizide',
    'glyburide'
]

# Encode categorical columns
le = LabelEncoder()
for col in cat_features:
    data[col] = data[col].astype(str).fillna('Unknown')
    data[col] = le.fit_transform(data[col])

# Combine all features
all_features = num_features + cat_features

# Final X and y
X = data[all_features].fillna(0)
y = data['readmitted_binary']

print(f"✅ Features ready  : {len(all_features)}")
print(f"📊 Training size   : {len(X):,} records")
print(f"\nFeatures used:")
for f in all_features:
    print(f"   • {f}")

✅ Features ready  : 20
📊 Training size   : 69,569 records

Features used:
   • age
   • time_in_hospital
   • num_lab_procedures
   • num_procedures
   • num_medications
   • number_outpatient
   • number_emergency
   • number_inpatient
   • number_diagnoses
   • admission_type_id
   • discharge_disposition_id
   • admission_source_id
   • race
   • gender
   • insulin
   • diabetesMed
   • change
   • metformin
   • glipizide
   • glyburide


In [ ]:
#REJECTED MODEL EXPERIMENTS (COMMENTED OUT)

# These cells contain earlier model iterations that were tested
# and rejected. They are kept for documentation and learning
# purposes but are NOT part of the final pipeline.
#
#
#
#
#
#
#



In [6]:
# OneHotEncoder experiment Rejected: OHE expanded features to 39 columns but caused high-risk recall to collapse to 7% at threshold 0.45. Needs threshold ~0.20 to work — flags 11,000+ patients, unworkable for care teams.

'''from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np
# ── Numerical features (used as-is) ──────────────────
num_features = [
    'age',
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

# ── Categorical features (need OneHotEncoding) ────────
cat_features = [
    'race',
    'gender',
    'insulin',
    'diabetesMed',
    'change',
    'metformin',
    'glipizide',
    'glyburide'
]

# ── Fill missing values before encoding ──────────────
for col in cat_features:
    data[col] = data[col].astype(str).fillna('Unknown')

# ── Build ColumnTransformer ───────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_features),
        ('cat', OneHotEncoder(
            handle_unknown='ignore',  # ignore unseen categories
            sparse_output=False       # return dense array
        ), cat_features)
    ]
)

# ── Fit and transform the data ────────────────────────
X_raw = data[num_features + cat_features].fillna(0)
X     = preprocessor.fit_transform(X_raw)

# ── Get feature names for interpretability ────────────
ohe_feature_names = preprocessor.named_transformers_['cat']\
                    .get_feature_names_out(cat_features).tolist()
all_features      = num_features + ohe_feature_names
y                 = data['readmitted_binary']

print(f"✅ Encoding       : OneHotEncoding")
print(f"✅ Original cols  : {len(num_features + cat_features)}")
print(f"✅ Expanded cols  : {len(all_features)} features after OHE")
print(f"📊 Dataset size   : {len(X):,} records")
print(f"\nSample OHE features created:")
for f in ohe_feature_names[:10]:
    print(f"   • {f}")
print(f"   ... and {len(ohe_feature_names)-10} more")

SyntaxError: incomplete input (1110492811.py, line 1)

In [ ]:
# Random split, no SMOTE (baseline) Accuracy: 71.35% | High Risk Recall: 42% Rejected: No SMOTE = model barely learned high-risk patterns. Random split mixes time periods.

'''# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("🤖 Training Random Forest...")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
model.fit(X_train, y_train)

# Evaluate
y_pred  = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Model trained!")
print(f"📊 Accuracy        : {accuracy*100:.2f}%")
print(f"\n--- Classification Report ---")
print(classification_report(y_test, y_pred,
      target_names=['Low/No Risk', 'High Risk (<30 days)']))

# Top features
print("--- Top 5 Most Predictive Features ---")
importance = pd.Series(model.feature_importances_, index=all_features)
print(importance.nlargest(5).to_string())'''

Rejected due to unbalanced sheet

In [ ]:
# Random split, full SMOTE (100% balance) Accuracy: 78.52% | High Risk Recall: 18% Rejected: Overfit SMOTE distorted class distribution. Model learned a fake 50/50 world.

'''from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

# Install imbalanced-learn
import subprocess
subprocess.run(['pip', 'install', 'imbalanced-learn', '-q'])

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ── SMOTE: Fix class imbalance ───────────────────────
# Creates synthetic high-risk patient samples
# You used SMOTE in your Climate Change ML project too!
print("⚖️  Balancing classes with SMOTE...")
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE → High Risk: {y_train.sum():,} | Low Risk: {(y_train==0).sum():,}")
print(f"After SMOTE  → High Risk: {y_train_bal.sum():,} | Low Risk: {(y_train_bal==0).sum():,}")

# ── Train Improved Model ─────────────────────────────
print("\n🤖 Training Random Forest...")
model = RandomForestClassifier(
    n_estimators=200,       # more trees = more stable
    max_depth=15,           # deeper = captures more patterns
    min_samples_leaf=5,     # avoids overfitting
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_bal, y_train_bal)

# ── Evaluate ─────────────────────────────────────────
y_pred   = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Model trained!")
print(f"📊 Accuracy        : {accuracy*100:.2f}%")
print(f"\n--- Classification Report ---")
print(classification_report(y_test, y_pred,
      target_names=['Low/No Risk', 'High Risk (<30 days)']))

# ── Top Features ──────────────────────────────────────
print("--- Top 5 Most Predictive Features ---")
importance = pd.Series(model.feature_importances_, index=all_features)
print(importance.nlargest(5).to_string())'''



Rejected

In [ ]:
# Random split, SMOTE 0.5, threshold 0.30 Accuracy: 31.51% | High Risk Recall: 81% Rejected: Catches 81% but flags 10,282 patients - no hospital care team can follow up on that many.

'''!pip install imbalanced-learn -q

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import numpy as np

# ── Split ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ── SMOTE ────────────────────────────────────────────
print("⚖️  Balancing with SMOTE...")
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print(f"✅ After SMOTE → High Risk: {y_train_bal.sum():,} | Low Risk: {(y_train_bal==0).sum():,}")

# ── Train Model ──────────────────────────────────────
print("\n🤖 Training Random Forest...")
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=3,
    class_weight={0: 1, 1: 3},  # penalize missing high-risk 3x more
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_bal, y_train_bal)

# ── Lower Threshold to 30% ───────────────────────────
# Default is 50% — too strict for catching high risk patients
# In healthcare, false alarm > missed patient
THRESHOLD = 0.30

y_proba  = model.predict_proba(X_test)[:, 1]
y_pred   = (y_proba >= THRESHOLD).astype(int)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Model trained!")
print(f"📊 Accuracy              : {accuracy*100:.2f}%")
print(f"🎯 Prediction Threshold  : {THRESHOLD} (lowered from 0.50)")
print(f"\n--- Classification Report ---")
print(classification_report(y_test, y_pred,
      target_names=['Low/No Risk', 'High Risk (<30 days)']))

# ── Show threshold impact ────────────────────────────
print("--- Threshold Comparison ---")
for t in [0.20, 0.30, 0.40, 0.50]:
    pred_t  = (y_proba >= t).astype(int)
    caught  = ((pred_t == 1) & (y_test == 1)).sum()
    total   = y_test.sum()
    flagged = pred_t.sum()
    print(f"  Threshold {t} → Caught {caught}/{total} high-risk "
          f"({caught/total*100:.0f}%) | Flagged {flagged} patients")

# ── Top Features ─────────────────────────────────────
print("\n--- Top 5 Most Predictive Features ---")
importance = pd.Series(model.feature_importances_, index=all_features)
print(importance.nlargest(5).to_string())'''

Rejected

In [ ]:
#Random split, SMOTE 0.5, threshold 0.45 Accuracy: 62.21% | High Risk Recall: 41% Rejected: Same model as final but uses random split instead of temporal - inflates performance.

'''!pip install imbalanced-learn -q

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import numpy as np

# ── Split ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ── SMOTE ────────────────────────────────────────────
print("⚖️  Balancing with SMOTE...")
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print(f"✅ After SMOTE → High Risk: {y_train_bal.sum():,} | Low Risk: {(y_train_bal==0).sum():,}")

# ── Train Model ──────────────────────────────────────
print("\n🤖 Training Random Forest...")
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=3,
    class_weight={0: 1, 1: 3},  # penalize missing high-risk 3x more
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_bal, y_train_bal)

# ── Lower Threshold to 30% ───────────────────────────
# Default is 50% — too strict for catching high risk patients
# In healthcare, false alarm > missed patient
THRESHOLD = 0.45

y_proba  = model.predict_proba(X_test)[:, 1]
y_pred   = (y_proba >= THRESHOLD).astype(int)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Model trained!")
print(f"📊 Accuracy              : {accuracy*100:.2f}%")
print(f"🎯 Prediction Threshold  : {THRESHOLD} (lowered from 0.50)")
print(f"\n--- Classification Report ---")
print(classification_report(y_test, y_pred,
      target_names=['Low/No Risk', 'High Risk (<30 days)']))

# ── Show threshold impact ────────────────────────────
print("--- Threshold Comparison ---")
for t in [0.20, 0.30, 0.40, 0.50]:
    pred_t  = (y_proba >= t).astype(int)
    caught  = ((pred_t == 1) & (y_test == 1)).sum()
    total   = y_test.sum()
    flagged = pred_t.sum()
    print(f"  Threshold {t} → Caught {caught}/{total} high-risk "
          f"({caught/total*100:.0f}%) | Flagged {flagged} patients")

# ── Top Features ─────────────────────────────────────
print("\n--- Top 5 Most Predictive Features ---")
importance = pd.Series(model.feature_importances_, index=all_features)
print(importance.nlargest(5).to_string())'''

Rejected

In [ ]:
# Temporal split, threshold 0.50 Accuracy: 70.17% | High Risk Recall: 30% Rejected: Threshold too strict — misses too many high-risk patients (only catches 274/920).

"""!pip install imbalanced-learn -q

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import numpy as np

# ── Temporal Split ───────────────────────────────────
data_sorted = df.copy()
data_sorted.replace('?', np.nan, inplace=True)

# Sort by encounter_id as time proxy
data_sorted = data_sorted.sort_values('encounter_id')

# Re-apply cleaning on sorted data
data_sorted.drop(columns=['weight','payer_code',
                           'medical_specialty'], inplace=True)
data_sorted.drop_duplicates(subset='patient_nbr',
                             keep='first', inplace=True)
data_sorted.dropna(subset=['race'], inplace=True)
data_sorted = data_sorted[
    data_sorted['gender'] != 'Unknown/Invalid']

age_map = {
    '[0-10)':5,   '[10-20)':15, '[20-30)':25,
    '[30-40)':35, '[40-50)':45, '[50-60)':55,
    '[60-70)':65, '[70-80)':75, '[80-90)':85,
    '[90-100)':95
}
data_sorted['age'] = data_sorted['age'].map(age_map)
data_sorted['readmitted_binary'] = (
    data_sorted['readmitted'] == '<30').astype(int)

# Encode categoricals
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cat_features = ['race','gender','insulin','diabetesMed',
                'change','metformin','glipizide','glyburide']
for col in cat_features:
    data_sorted[col] = data_sorted[col].astype(str).fillna('Unknown')
    data_sorted[col] = le.fit_transform(data_sorted[col])

num_features = [
    'age','time_in_hospital','num_lab_procedures',
    'num_procedures','num_medications','number_outpatient',
    'number_emergency','number_inpatient','number_diagnoses',
    'admission_type_id','discharge_disposition_id',
    'admission_source_id'
]
all_features = num_features + cat_features

# ── 80/20 Temporal Split ─────────────────────────────
split_idx  = int(len(data_sorted) * 0.80)
train_data = data_sorted.iloc[:split_idx]
test_data  = data_sorted.iloc[split_idx:]

X_train = train_data[all_features].fillna(0)
y_train = train_data['readmitted_binary']
X_test  = test_data[all_features].fillna(0)
y_test  = test_data['readmitted_binary']

print(f"📋 Training set : {len(X_train):,} records (earlier)")
print(f"📋 Test set     : {len(X_test):,}  records (later)")
print(f"\nTraining High Risk: {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"Test High Risk    : {y_test.sum():,}  ({y_test.mean()*100:.1f}%)")

# ── SMOTE on training only ───────────────────────────
# ⚠️ NEVER apply SMOTE to test data — only training!
print("\n⚖️  Applying SMOTE to training set only...")
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"After SMOTE → High: {y_train_bal.sum():,} | "
      f"Low: {(y_train_bal==0).sum():,}")

# ── Train Model ──────────────────────────────────────
print("\n🤖 Training Random Forest...")
model_temporal = RandomForestClassifier(
    n_estimators  = 200,
    max_depth     = 12,
    min_samples_leaf = 3,
    class_weight  = {0:1, 1:3},
    random_state  = 42,
    n_jobs        = -1
)
model_temporal.fit(X_train_bal, y_train_bal)

# ── Evaluate with 0.50 threshold ─────────────────────
THRESHOLD = 0.50
y_proba   = model_temporal.predict_proba(X_test)[:, 1]
y_pred    = (y_proba >= THRESHOLD).astype(int)
accuracy  = accuracy_score(y_test, y_pred)

print(f"\n✅ Temporal Validation Results:")
print(f"📊 Accuracy    : {accuracy*100:.2f}%")
print(f"🎯 Threshold   : {THRESHOLD}")
print(f"\n--- Classification Report ---")
print(classification_report(y_test, y_pred,
      target_names=['Low/No Risk', 'High Risk (<30 days)']))

# ── Threshold comparison ──────────────────────────────
print("--- Threshold Comparison ---")
for t in [0.20, 0.30, 0.40, 0.45, 0.50]:
    pred_t  = (y_proba >= t).astype(int)
    caught  = ((pred_t==1) & (y_test==1)).sum()
    total   = y_test.sum()
    flagged = pred_t.sum()
    print(f"  Threshold {t} → Caught {caught}/{total} "
          f"({caught/total*100:.0f}%) | "
          f"Flagged {flagged:,} patients")

# ── Feature importance ────────────────────────────────
print("\n--- Top 5 Most Predictive Features ---")
importance = pd.Series(model_temporal.feature_importances_,
                       index=all_features)
print(importance.nlargest(5).to_string())"""


Rejected

In [7]:
# FINAL DECISION: selected as best model.
# Reason: Temporal split + SMOTE on training only + threshold
# 0.45 gives the most honest and clinically deployable result.
# PURPOSE:
# Train and evaluate the final production-ready readmission
# risk prediction model using best practices for healthcare ML.

!pip install imbalanced-learn -q

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import numpy as np

# Temporal Split
data_sorted = df.copy()
data_sorted.replace('?', np.nan, inplace=True)

# Sort by encounter_id as time proxy
data_sorted = data_sorted.sort_values('encounter_id')

# Re-apply cleaning on sorted data
data_sorted.drop(columns=['weight','payer_code',
                           'medical_specialty'], inplace=True)
data_sorted.drop_duplicates(subset='patient_nbr',
                             keep='first', inplace=True)
data_sorted.dropna(subset=['race'], inplace=True)
data_sorted = data_sorted[
    data_sorted['gender'] != 'Unknown/Invalid']

age_map = {
    '[0-10)':5,   '[10-20)':15, '[20-30)':25,
    '[30-40)':35, '[40-50)':45, '[50-60)':55,
    '[60-70)':65, '[70-80)':75, '[80-90)':85,
    '[90-100)':95
}
data_sorted['age'] = data_sorted['age'].map(age_map)
data_sorted['readmitted_binary'] = (
    data_sorted['readmitted'] == '<30').astype(int)

# Encode categoricals
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cat_features = ['race','gender','insulin','diabetesMed',
                'change','metformin','glipizide','glyburide']
for col in cat_features:
    data_sorted[col] = data_sorted[col].astype(str).fillna('Unknown')
    data_sorted[col] = le.fit_transform(data_sorted[col])

num_features = [
    'age','time_in_hospital','num_lab_procedures',
    'num_procedures','num_medications','number_outpatient',
    'number_emergency','number_inpatient','number_diagnoses',
    'admission_type_id','discharge_disposition_id',
    'admission_source_id'
]
all_features = num_features + cat_features

# 80/20 Temporal Split
split_idx  = int(len(data_sorted) * 0.80)
train_data = data_sorted.iloc[:split_idx]
test_data  = data_sorted.iloc[split_idx:]

X_train = train_data[all_features].fillna(0)
y_train = train_data['readmitted_binary']
X_test  = test_data[all_features].fillna(0)
y_test  = test_data['readmitted_binary']

print(f"📋 Training set : {len(X_train):,} records (earlier)")
print(f"📋 Test set     : {len(X_test):,}  records (later)")
print(f"\nTraining High Risk: {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"Test High Risk    : {y_test.sum():,}  ({y_test.mean()*100:.1f}%)")

#  SMOTE on training only
#NEVER apply SMOTE to test data - only training!
print("\n⚖️  Applying SMOTE to training set only...")
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"After SMOTE → High: {y_train_bal.sum():,} | "
      f"Low: {(y_train_bal==0).sum():,}")

#  Train Model
print("\n🤖 Training Random Forest...")
model_temporal = RandomForestClassifier(
    n_estimators  = 200,
    max_depth     = 12,
    min_samples_leaf = 3,
    class_weight  = {0:1, 1:3},
    random_state  = 42,
    n_jobs        = -1
)
model_temporal.fit(X_train_bal, y_train_bal)

#  Evaluate with 0.45 threshold
THRESHOLD = 0.45
y_proba   = model_temporal.predict_proba(X_test)[:, 1]
y_pred    = (y_proba >= THRESHOLD).astype(int)
accuracy  = accuracy_score(y_test, y_pred)

print(f"\n✅ Temporal Validation Results:")
print(f"📊 Accuracy    : {accuracy*100:.2f}%")
print(f"🎯 Threshold   : {THRESHOLD}")
print(f"\n--- Classification Report ---")
print(classification_report(y_test, y_pred,
      target_names=['Low/No Risk', 'High Risk (<30 days)']))

#  Threshold comparison
print("--- Threshold Comparison ---")
for t in [0.20, 0.30, 0.40, 0.45, 0.50]:
    pred_t  = (y_proba >= t).astype(int)
    caught  = ((pred_t==1) & (y_test==1)).sum()
    total   = y_test.sum()
    flagged = pred_t.sum()
    print(f"  Threshold {t} → Caught {caught}/{total} "
          f"({caught/total*100:.0f}%) | "
          f"Flagged {flagged:,} patients")

#  Feature importance
print("\n--- Top 5 Most Predictive Features ---")
importance = pd.Series(model_temporal.feature_importances_,
                       index=all_features)
print(importance.nlargest(5).to_string())


📋 Training set : 55,655 records (earlier)
📋 Test set     : 13,914  records (later)

Training High Risk: 5,232 (9.4%)
Test High Risk    : 920  (6.6%)

⚖️  Applying SMOTE to training set only...
After SMOTE → High: 25,211 | Low: 50,423

🤖 Training Random Forest...

✅ Temporal Validation Results:
📊 Accuracy    : 62.38%
🎯 Threshold   : 0.45

--- Classification Report ---
                      precision    recall  f1-score   support

         Low/No Risk       0.94      0.64      0.76     12994
High Risk (<30 days)       0.07      0.40      0.12       920

            accuracy                           0.62     13914
           macro avg       0.51      0.52      0.44     13914
        weighted avg       0.88      0.62      0.72     13914

--- Threshold Comparison ---
  Threshold 0.2 → Caught 878/920 (95%) | Flagged 13,064 patients
  Threshold 0.3 → Caught 746/920 (81%) | Flagged 10,317 patients
  Threshold 0.4 → Caught 476/920 (52%) | Flagged 6,653 patients
  Threshold 0.45 → Caught 367/92

In [8]:
# Run this quick check first. PURPOSE: Quick sanity check to confirm the correct model, dataset, and feature set are loaded in memory before scoring patients.
print(f"✅ Model  : {type(model_temporal).__name__}")
print(f"✅ Records: {len(data):,}")
print(f"✅ Features: {len(all_features)}")

✅ Model  : RandomForestClassifier
✅ Records: 69,569
✅ Features: 20


In [11]:
# Score all patients with THRESHOLD = 0.45
# PURPOSE: Apply the trained model to ALL patients and assign each one a risk score, risk level, and recommended clinical action.
# HOW IT WORKS: model_temporal.predict_proba() returns a probability between 0 and 1 for each patient. We multiply by 100 for percentage.
THRESHOLD = 0.45

# Use model_temporal (your best model)
data_sorted['risk_score'] = model_temporal.predict_proba(
    data_sorted[all_features].fillna(0))[:, 1] * 100

# RISK LEVEL ASSIGNMENT: Score ≥ 70% → 🔴 High Risk   → Call patient within 24 hours, Score 45-70% → 🟡 Medium Risk → Schedule follow-up in 3 days
# Score < 45%  → 🟢 Low Risk   → Standard discharge care plan

# Risk levels
def assign_risk(score):
    if score >= 70:
        return '🔴 High Risk'
    elif score >= 45:
        return '🟡 Medium Risk'
    else:
        return '🟢 Low Risk'

# Recommended actions
def assign_action(score):
    if score >= 70:
        return 'Call patient within 24 hours'
    elif score >= 45:
        return 'Schedule follow-up within 3 days'
    else:
        return 'Standard discharge care plan'

data_sorted['risk_level']         = data_sorted['risk_score'].apply(assign_risk)
data_sorted['recommended_action'] = data_sorted['risk_score'].apply(assign_action)

# Summary
high   = (data_sorted['risk_score'] >= 70).sum()
medium = data_sorted['risk_score'].between(45, 70).sum()
low    = (data_sorted['risk_score'] < 45).sum()

print("📊 Final Risk Summary (Threshold = 0.45):")
print(f"   🔴 High Risk    : {high:,} → Call within 24hrs")
print(f"   🟡 Medium Risk  : {medium:,} → Follow-up in 3 days")
print(f"   🟢 Low Risk     : {low:,} → Standard care")
print(f"\nTop 5 Highest Risk Patients:")
print(data_sorted[data_sorted['risk_score'] >= 70][
    ['patient_nbr','age','risk_score',
     'risk_level','recommended_action']].head(5))

📊 Final Risk Summary (Threshold = 0.45):
   🔴 High Risk    : 4,831 → Call within 24hrs
   🟡 Medium Risk  : 21,909 → Follow-up in 3 days
   🟢 Low Risk     : 42,829 → Standard care

Top 5 Highest Risk Patients:
    patient_nbr  age  risk_score   risk_level            recommended_action
9      63555939   95   82.301247  🔴 High Risk  Call patient within 24 hours
31     96664626   75   73.115147  🔴 High Risk  Call patient within 24 hours
34      3327282   75   74.491920  🔴 High Risk  Call patient within 24 hours
35     63023292   65   81.741298  🔴 High Risk  Call patient within 24 hours
46     86240259   75   77.021944  🔴 High Risk  Call patient within 24 hours


In [12]:
# Generate Excel report file
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import date

today    = date.today().strftime("%B %d, %Y")
filename = f"Readmission_Risk_Report_{date.today()}.xlsx"

#  Build Report DataFrame
report = data_sorted[[
    'patient_nbr',
    'age',
    'time_in_hospital',
    'num_medications',
    'number_inpatient',
    'number_emergency',
    'number_diagnoses',
    'risk_score',
    'risk_level',
    'recommended_action'
]].copy()

report.columns = [
    'Patient ID',
    'Age',
    'Length of Stay (Days)',
    'Medications',
    'Previous Admissions',
    'Emergency Visits',
    'Number of Diagnoses',
    'Risk Score (%)',
    'Risk Level',
    'Recommended Action'
]

# Round and sort by highest risk first
report['Risk Score (%)'] = report['Risk Score (%)'].round(1)
report = report.sort_values('Risk Score (%)', ascending=False)

#  Save with Color Formatting
with pd.ExcelWriter(filename, engine='openpyxl') as writer:
    report.to_excel(writer, index=False, sheet_name='Risk Report')
    ws = writer.sheets['Risk Report']

    # Header styling - dark background
    header_fill = PatternFill("solid", fgColor="1a1a2e")
    header_font = Font(color="FFFFFF", bold=True, size=11)
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = Alignment(horizontal='center')

    # Row color coding by risk level
    red_fill    = PatternFill("solid", fgColor="FFE0E0")
    yellow_fill = PatternFill("solid", fgColor="FFF9C4")
    green_fill  = PatternFill("solid", fgColor="E0F7E9")

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        risk = row[8].value
        if risk and 'High' in risk:
            fill = red_fill
        elif risk and 'Medium' in risk:
            fill = yellow_fill
        else:
            fill = green_fill
        for cell in row:
            cell.fill = fill

    # Auto column widths
    for col in ws.columns:
        max_len = max(
            len(str(cell.value)) if cell.value else 0
            for cell in col
        )
        ws.column_dimensions[
            get_column_letter(col[0].column)].width = max_len + 4

# Summary
high   = (report['Risk Score (%)'] >= 70).sum()
medium = report['Risk Score (%)'].between(45, 70).sum()
low    = (report['Risk Score (%)'] < 45).sum()

print(f"✅ Report saved   : {filename}")
print(f"📋 Total patients : {len(report):,}")
print(f"\n📊 Summary:")
print(f"   🔴 High Risk   : {high:,}")
print(f"   🟡 Medium Risk : {medium:,}")
print(f"   🟢 Low Risk    : {low:,}")
print(f"\nTop 5 Highest Risk Patients:")
print(report.head()[['Patient ID','Age',
                      'Risk Score (%)','Risk Level',
                      'Recommended Action']])

✅ Report saved   : Readmission_Risk_Report_2026-03-17.xlsx
📋 Total patients : 69,569

📊 Summary:
   🔴 High Risk   : 4,841
   🟡 Medium Risk : 22,010
   🟢 Low Risk    : 42,751

Top 5 Highest Risk Patients:
       Patient ID  Age  Risk Score (%)   Risk Level  \
20626   103712364   85            93.0  🔴 High Risk   
51936   102236301   75            92.5  🔴 High Risk   
26383    19852668   65            92.2  🔴 High Risk   
19315     1055151   95            91.9  🔴 High Risk   
32411     9165366   85            91.7  🔴 High Risk   

                 Recommended Action  
20626  Call patient within 24 hours  
51936  Call patient within 24 hours  
26383  Call patient within 24 hours  
19315  Call patient within 24 hours  
32411  Call patient within 24 hours  


In [15]:
#GMAIL CREDENTIALS for sending report to authorized personnel
GMAIL_USER     = "your_email@gmail.com"
GMAIL_PASSWORD = "email app password"
RECIPIENT      = "recipient_email@gmail.com"

In [16]:
#AUTOMATED EMAIL DELIVERY

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from datetime import date

def send_risk_report(filename):
    today_str = date.today().strftime("%B %d, %Y")
    high   = (report['Risk Score (%)'] >= 70).sum()
    medium = report['Risk Score (%)'].between(45, 70).sum()
    low    = (report['Risk Score (%)'] < 45).sum()
    total  = len(report)

    msg            = MIMEMultipart("mixed")
    msg['From']    = GMAIL_USER
    msg['To']      = RECIPIENT
    msg['Subject'] = f"🏥 Daily Readmission Risk Report — {today_str}"

    html = f"""
    <html>
    <body style="font-family: Arial, sans-serif; max-width:600px;
                 margin:auto; padding:30px; color:#222;">

        <h2>🏥 Daily Readmission Risk Report</h2>
        <p style="color:#666;">{today_str}</p>
        <hr>
        <p>Patient readmission risk summary
           across <strong>{total:,} patients</strong>:</p>

        <table style="width:100%; border-collapse:collapse; margin:20px 0;">
            <tr style="background:#fee2e2;">
                <td style="padding:12px;">🔴 High Risk</td>
                <td style="padding:12px; font-weight:bold;
                           font-size:18px;">{high:,}</td>
                <td style="padding:12px; color:#666;
                           font-size:13px;">Call within 24 hours</td>
            </tr>
            <tr style="background:#fef9c3;">
                <td style="padding:12px;">🟡 Medium Risk</td>
                <td style="padding:12px; font-weight:bold;
                           font-size:18px;">{medium:,}</td>
                <td style="padding:12px; color:#666;
                           font-size:13px;">Follow-up in 3 days</td>
            </tr>
            <tr style="background:#dcfce7;">
                <td style="padding:12px;">🟢 Low Risk</td>
                <td style="padding:12px; font-weight:bold;
                           font-size:18px;">{low:,}</td>
                <td style="padding:12px; color:#666;
                           font-size:13px;">Standard care plan</td>
            </tr>
        </table>

        <p>Model: Random Forest + SMOTE + Temporal Validation<br>
           Accuracy: 62.38% | High Risk Recall: 40%<br>
           Top predictors: medication change, metformin, insulin</p>

        <p>Full color-coded patient report attached.</p>
        <hr>
        <p style="font-size:13px; color:#888;">
            Built by <strong>Yashwant Katailiha</strong> |
            <a href="https://linkedin.com/in/ysk26"
               style="color:#4F46E5;">linkedin.com/in/ysk26</a> |
            <a href="https://github.com/ysk26"
               style="color:#4F46E5;">github.com/ysk26</a>
        </p>
    </body>
    </html>
    """

    body = MIMEMultipart("alternative")
    body.attach(MIMEText(html, 'html'))
    msg.attach(body)

    # Attach Excel
    with open(filename, "rb") as f:
        att = MIMEBase("application", "octet-stream")
        att.set_payload(f.read())
        encoders.encode_base64(att)
        att.add_header("Content-Disposition",
                       f"attachment; filename={filename}")
        msg.attach(att)

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(GMAIL_USER, GMAIL_PASSWORD)
            server.send_message(msg)
            print(f"✅ Report emailed to {RECIPIENT}!")
            print(f"📎 Attached: {filename}")
    except Exception as e:
        print(f"❌ Email failed: {e}")

send_risk_report(filename)

✅ Report emailed to vivekpandeyapkp@gmail.com!
📎 Attached: Readmission_Risk_Report_2026-03-17.xlsx


In [ ]:
#Download copy on local computer for records

from google.colab import files
files.download(filename)
print(f"✅ Downloaded: {filename}")
